# Linear Regression

Modela a relacao entre features `X` e um alvo continuo `y` como:

$$\hat{y} = X w + b$$

Objetivo: minimizar o erro quadratico medio (MSE).

Duas formas de resolver:
- **Equacao normal**: solucao fechada, rapida para poucas features.
- **Gradient descent**: iterativo, escalavel e extensivel a regularizacao.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
class LinearRegression:
    """
    Regressao linear ordinaria.

    Suporta dois metodos de ajuste:
      - "normal": solucao fechada via equacao normal (X^T X)^-1 X^T y,
        com fallback para pinv quando a matriz e singular.
      - "gd": gradient descent em batch, util quando ha muitas features
        ou quando se quer usar regularizacao explicita no futuro.
    """

    def __init__(self, method="normal", lr=0.01, n_iters=1000, fit_intercept=True):
        if method not in {"normal", "gd"}:
            raise ValueError("method must be 'normal' or 'gd'.")
        if lr <= 0:
            raise ValueError("lr must be positive.")
        if n_iters <= 0:
            raise ValueError("n_iters must be positive.")

        self.method = method
        self.lr = lr
        self.n_iters = n_iters
        self.fit_intercept = fit_intercept

        self.coef_ = None
        self.intercept_ = 0.0
        self.loss_history_ = []

    def _validate_X_y(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()

        if X.ndim != 2:
            raise ValueError("X must be 2D with shape (n_samples, n_features).")
        if len(X) != len(y):
            raise ValueError("X and y must have the same number of samples.")
        if len(X) == 0:
            raise ValueError("Training data cannot be empty.")
        if np.isnan(X).any() or np.isnan(y).any():
            raise ValueError("NaN values detected. Handle missing values first.")

        return X, y

    def _validate_X(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if self.coef_ is None:
            raise ValueError("Call fit() before predict().")
        if X.shape[1] != self.coef_.shape[0]:
            raise ValueError("X has a different number of features than training data.")
        return X

    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def fit(self, X, y):
        X, y = self._validate_X_y(X, y)
        n_samples, n_features = X.shape

        if self.method == "normal":
            Xb = self._add_bias(X) if self.fit_intercept else X
            try:
                theta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
            except np.linalg.LinAlgError:
                # matriz singular: fallback para pseudo-inversa
                theta = np.linalg.pinv(Xb) @ y

            if self.fit_intercept:
                self.intercept_ = float(theta[0])
                self.coef_ = theta[1:]
            else:
                self.intercept_ = 0.0
                self.coef_ = theta
            return self

        # gradient descent
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0.0
        self.loss_history_ = []

        for _ in range(self.n_iters):
            y_pred = X @ self.coef_ + self.intercept_
            error = y_pred - y

            grad_w = (2.0 / n_samples) * (X.T @ error)
            grad_b = (2.0 / n_samples) * error.sum()

            self.coef_ -= self.lr * grad_w
            if self.fit_intercept:
                self.intercept_ -= self.lr * grad_b

            self.loss_history_.append(float(np.mean(error ** 2)))

        return self

    def predict(self, X):
        X = self._validate_X(X)
        return X @ self.coef_ + self.intercept_

    def score(self, X, y):
        """Coeficiente de determinacao R^2."""
        y = np.asarray(y, dtype=float).ravel()
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        if ss_tot == 0:
            return 0.0
        return 1.0 - ss_res / ss_tot

In [ ]:
X, y = datasets.make_regression(
    n_samples=200, n_features=1, noise=15, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression(method="gd", lr=0.05, n_iters=1500)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"coef:      {model.coef_}")
print(f"intercept: {model.intercept_:.4f}")
print(f"MSE:       {mean_squared_error(y_test, y_pred):.4f}")
print(f"R2:        {r2_score(y_test, y_pred):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_train, y_train, color="steelblue", s=20, label="train")
axes[0].scatter(X_test, y_test, color="orange", s=20, label="test")
xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
axes[0].plot(xs, model.predict(xs), color="black", linewidth=2, label="fit")
axes[0].set_title("Linear Regression fit")
axes[0].legend()

axes[1].plot(model.loss_history_)
axes[1].set_title("MSE over iterations")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("MSE")

plt.tight_layout()
plt.show()